# Temporal decay fit

This plots the per-pixel fit of $\gamma(t) = \gamma_\infty + (1 - \gamma_\infty)e^{-t/\tau}$ made by `scripts/fit_decay.py` (submitted with `slurm/fit_decay.sbatch`).

In [ ]:
import numpy as np
import xarray as xr
import rioxarray  # noqa: F401
import matplotlib.pyplot as plt
from seaborn import set_theme
set_theme()

import sys
import os
from pathlib import Path

# import from this repo
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

%load_ext autoreload
%autoreload 2
from scripts.taus import decorrelation_temporal_model

Open the output Zarr of the fit. The selection that was used (variable, dates, tracks) is stored in its attributes.

In [ ]:
data_dir = "../data/"
res = '80m'

fit = xr.open_zarr(data_dir + f'nisar/zarrs/decay_{res}.zarr', consolidated=False).load()
{k: fit.attrs[k] for k in ['var', 'gamma_inf', 'start_date', 'end_date', 'max_days', 'inputs']} | {'n_pairs_selected': len(fit.attrs['pairs'])}

Plot $\tau$ and $\gamma_\infty$. Pixels that were not fit (too few valid pairs, or no convergence) are blank.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
fit['tau'].plot(ax=axs[0], cmap='viridis', robust=True, cbar_kwargs={'label': 'days'})
axs[0].set_title(r'$\tau$')
fit['gamma_inf'].plot(ax=axs[1], cmap='inferno', vmin=0, vmax=1)
axs[1].set_title(r'$\gamma_\infty$')
for ax in axs:
    ax.set_aspect('equal')
plt.tight_layout()

Plot the 1-sigma fit errors, to see where the fit is poorly constrained. $\gamma_\infty$ error is empty if $\gamma_\infty$ was fixed.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
fit['tau_err'].plot(ax=axs[0], cmap='magma', robust=True, cbar_kwargs={'label': 'days'})
axs[0].set_title(r'$\tau$ error')
fit['gamma_inf_err'].plot(ax=axs[1], cmap='magma', robust=True)
axs[1].set_title(r'$\gamma_\infty$ error')
for ax in axs:
    ax.set_aspect('equal')
plt.tight_layout()

Plot how many pairs went into each pixel and where the fit succeeded. Zero pairs inside a tile that ran means no data there; zero everywhere in a tile means the tile never ran.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
fit['n_pairs'].plot(ax=axs[0], cmap='cividis', cbar_kwargs={'label': 'pairs'})
axs[0].set_title('valid pairs')
fit['success'].astype(int).plot(ax=axs[1], cmap='gray_r', add_colorbar=False)
axs[1].set_title(f"fit succeeded ({float(fit['success'].mean()) * 100:.0f}% of pixels)")
for ax in axs:
    ax.set_aspect('equal')
plt.tight_layout()

Histograms of the fitted parameters over all successful pixels.

In [ ]:
ok = fit['success'].values
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].hist(fit['tau'].values[ok], bins=60)
axs[0].set_xlabel(r'$\tau$ (days)')
axs[1].hist(fit['gamma_inf'].values[ok], bins=60)
axs[1].set_xlabel(r'$\gamma_\infty$')
plt.tight_layout()

Check the fit at one pixel against its coherence time series. Pick a pixel by its x and y coordinates.

In [ ]:
from scripts.fit_decay import open_stack

# same selection as the fit; open_stack applies the date window and stacks the tracks
cube, days, _ = open_stack([Path(p) for p in fit.attrs['inputs']], fit.attrs['var'],
                           fit.attrs['start_date'] or None, fit.attrs['end_date'] or None,
                           fit.attrs['max_days'] or None)

px = dict(x=fit['x'].values[len(fit['x']) // 2], y=fit['y'].values[len(fit['y']) // 2])
obs = cube.sel(**px, method='nearest').values
p = fit.sel(**px, method='nearest')

t = np.linspace(0, days.max(), 200)
plt.scatter(days, obs, label='observed')
plt.plot(t, decorrelation_temporal_model(t, float(p['gamma_inf']), float(p['tau'])), 'r', label='fit')
plt.xlabel('days between acquisitions')
plt.ylabel(fit.attrs['var'])
plt.title(rf"$\tau$={float(p['tau']):.1f} d, $\gamma_\infty$={float(p['gamma_inf']):.2f}, {int(p['n_pairs'])} pairs")
plt.legend();